# Phase 2 — Predictions Table + AWS Bedrock Insights

Implements the second half of Phase 2 from the project blueprint:

1. **Predictions table** — train the tuned XGBoost model on all data available before the most recent completed fantasy week, generate projections for that week, and save them (with the context features that drove them) to a predictions table. In Databricks this would be a Delta table synced to Lakebase for the FastAPI layer; locally it is `data/predictions.parquet`.
2. **Bedrock integration (RAG flow)** — for each player: *retrieve* the model's projection + supporting stats from the predictions table, *augment* a prompt with them, and *generate* a qualitative insight with **GPT-5.6 Terra on AWS Bedrock** (`openai.gpt-5.6-terra`).

**API note:** GPT-5.6 models on Bedrock do *not* support the Converse/Invoke APIs. They are served exclusively through the OpenAI-compatible **Responses API** on the `bedrock-mantle` endpoint (`https://bedrock-mantle.{region}.api.aws/openai/v1`), authenticated with a bearer token generated from your AWS credentials. Terra is available in `us-east-1`, `us-east-2`, and `us-west-2`.

**Runs without AWS too:** if credentials are missing or the Bedrock call fails, the notebook falls back to a deterministic template-based insight generator so the full pipeline (predictions → insights → saved table) still executes end to end. Configure AWS credentials (`aws configure`) and request model access to GPT-5.6 Terra in the Bedrock console — no code changes needed.

In [22]:
# Install dependencies. GPT-5.6 models on Bedrock are served through the
# OpenAI-compatible Responses API, so we need the OpenAI SDK plus the AWS
# bearer-token generator (which reads standard AWS credentials).
%pip install xgboost scikit-learn openai aws-bedrock-token-generator

Note: you may need to restart the kernel to use updated packages.


## Build the Predictions Table

Same preprocessing and leakage guard as the model-eval notebooks. The model trains on everything strictly before the target week (all prior seasons + current season up to W−2), early-stops on week W−1, and projects week W — the exact production fold layout validated in `walk_forward_model.ipynb`.

In [23]:
import os
import pandas as pd
import numpy as np
from xgboost import XGBRegressor

# In Databricks: %run ../ingestion/feature_building
if 'gold_df' not in globals():
    gold_df = pd.read_parquet('../data/gold_df.parquet')

gold_df = gold_df[gold_df['week'] <= 17]  # fantasy season only
TARGET = 'fantasy_points_ppr'

eval_meta = gold_df[['player_id', 'player_name', 'recent_team', 'position',
                     'season', 'week', 'opponent']].copy()

df = gold_df.copy()
identifier_cols = ['player_id', 'player_name', 'recent_team', 'opponent', 'starting_qb_id', 'gameday']
df = df.drop(columns=[c for c in identifier_cols if c in df.columns])

same_week_outcome_cols = [
    'pass_attempts', 'completions', 'passing_yards', 'passing_tds', 'interceptions',
    'rush_attempts', 'rushing_yards', 'rushing_tds',
    'targets', 'receptions', 'receiving_yards', 'receiving_tds',
    'player_opportunities', 'team_total_opportunities', 'opportunity_share',
    'hvt_carries', 'hvt_targets', 'total_hvts',
    'team_pass_attempts', 'target_share',
    'player_air_yards', 'team_air_yards', 'air_yards_share',
    'wopr', 'snap_share',
]
df = df.drop(columns=[c for c in same_week_outcome_cols if c in df.columns])
df['position'] = eval_meta['position']
df = pd.get_dummies(df.fillna(0), columns=['position'], prefix='pos')
feature_cols = [c for c in df.columns if c != TARGET]

# Project the most recent completed fantasy week
PREDICT_SEASON = int(eval_meta['season'].max())
in_season = eval_meta['season'] == PREDICT_SEASON
PREDICT_WEEK = int(eval_meta.loc[in_season, 'week'].max())

test_mask = in_season & (eval_meta['week'] == PREDICT_WEEK)
val_mask = in_season & (eval_meta['week'] == PREDICT_WEEK - 1)
train_mask = ~in_season | (eval_meta['week'] <= PREDICT_WEEK - 2)

# Hyperparameters selected by the leakage-safe tuning in walk_forward_model.ipynb
model = XGBRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=4, min_child_weight=5,
    subsample=0.8, colsample_bytree=0.8,
    early_stopping_rounds=20, random_state=42, n_jobs=-1,
)
model.fit(
    df.loc[train_mask, feature_cols], df.loc[train_mask, TARGET],
    eval_set=[(df.loc[val_mask, feature_cols], df.loc[val_mask, TARGET])],
    verbose=False,
)
preds = np.clip(model.predict(df.loc[test_mask, feature_cols]), 0, None)

# Predictions table: projection + the context that drove it (for RAG retrieval)
context_cols = ['implied_total', 'team_spread', 'team_win_prob', 'is_home',
                'temp', 'wind', 'is_bad_weather', 'is_dome',
                'fantasy_points_3wk_avg', 'depth_chart_rank',
                'opp_def_ppg_allowed', 'prev_season_ppg']
predictions_df = eval_meta.loc[test_mask].reset_index(drop=True)
predictions_df['projected_ppr'] = preds.astype(float).round(1)
predictions_df['actual_ppr'] = df.loc[test_mask, TARGET].astype(float).round(1).values  # retrospective run: week already played
predictions_df = pd.concat(
    [predictions_df, df.loc[test_mask, context_cols].round(2).reset_index(drop=True)], axis=1
)
predictions_df = predictions_df.sort_values('projected_ppr', ascending=False).reset_index(drop=True)

os.makedirs('../data', exist_ok=True)
predictions_df.to_parquet('../data/predictions.parquet', index=False)

print(f"Projected {PREDICT_SEASON} week {PREDICT_WEEK}: {len(predictions_df)} players")
print(f"Saved predictions table to ../data/predictions.parquet")
display(predictions_df.head(10)[['player_name', 'recent_team', 'position', 'opponent',
                                 'projected_ppr', 'actual_ppr', 'implied_total']])

Projected 2025 week 17: 269 players
Saved predictions table to ../data/predictions.parquet


,player_name,recent_team,position,opponent,projected_ppr,actual_ppr,implied_total
0,P.Nacua,LA,WR,ATL,21.6,15.7,28.00
1,D.Maye,NE,QB,NYJ,20.7,32.4,27.50
2,J.Gibbs,DET,RB,MIN,20.3,8.4,26.25
3,C.McCaffrey,SF,RB,CHI,19.9,28.1,27.50
4,M.Stafford,LA,QB,ATL,19.7,11.9,28.00
5,B.Purdy,SF,QB,CHI,19.7,36.6,27.50
6,J.Allen,BUF,QB,PHI,19.3,23.2,24.25
7,T.Lawrence,JAX,QB,IND,19.3,22.6,26.00
8,J.Chase,CIN,WR,ARI,19.1,25.0,29.75
9,B.Nix,DEN,QB,KC,19.0,19.3,25.50


## RAG Flow: Retrieve → Augment → Generate

For each player the prompt is *augmented* with everything retrieved from the predictions table — projection, recent form, Vegas context, matchup, weather — so the LLM writes from **our data**, not its stale training knowledge. This is the pattern described in the project blueprint (the CeeDee Lamb example).

Insights are generated for the top 15 projected players to bound cost; in production this would run for the full slate.

In [24]:
# ============================================================================
# RAG: RETRIEVE stats -> AUGMENT prompt -> GENERATE insight via Bedrock
# ============================================================================
# GPT-5.6 Terra is served ONLY through the OpenAI-compatible Responses API on
# the bedrock-mantle endpoint — Converse/Invoke do not work for GPT-5.6 models.
BEDROCK_MODEL_ID = 'openai.gpt-5.6-terra'
# Fallback while Terra's marketplace subscription finishes provisioning:
# gpt-oss-120b is an OpenAI open-weight model that supports the Converse API.
FALLBACK_MODEL_ID = 'openai.gpt-oss-120b-1:0'
BEDROCK_REGION = os.environ.get('AWS_REGION', 'us-east-1')  # Terra: us-east-1 / us-east-2 / us-west-2
N_INSIGHTS = 15


def build_prompt(row):
    """AUGMENT step: inject retrieved model output + context into the prompt."""
    venue = 'at home' if row['is_home'] else 'on the road'
    weather = ('indoors (dome)' if row['is_dome']
               else f"{row['temp']:.0f}F, {row['wind']:.0f} mph wind"
               + (' — bad weather game' if row['is_bad_weather'] else ''))
    return f"""You are a fantasy football analyst. Using ONLY the data below, write a 2-3 sentence
insight for this player's upcoming game. Mention the projection, one supporting factor,
and one risk factor. Do not invent injuries or news not present in the data.

Player: {row['player_name']} ({row['position']}, {row['recent_team']})
Opponent: {row['opponent']} ({venue})
Model projection: {row['projected_ppr']} PPR points
Recent form (3-week avg): {row['fantasy_points_3wk_avg']} PPR points
Previous season average: {row['prev_season_ppg']} PPR points
Vegas implied team total: {row['implied_total']} | spread: {row['team_spread']:+.1f} | win prob: {row['team_win_prob']:.0%}
Opponent defense allows {row['opp_def_ppg_allowed']} PPR points/game to {row['position']}s
Depth chart rank: {int(row['depth_chart_rank'])}
Weather: {weather}"""


def template_insight(row):
    """Offline fallback so the pipeline runs end-to-end without AWS credentials."""
    lean = 'favorable' if row['team_spread'] > 0 else 'tough'
    parts = [
        f"{row['player_name']} projects for {row['projected_ppr']} PPR points against {row['opponent']}.",
        f"Vegas implies a {row['implied_total']:.1f}-point team total in a {lean} game script, "
        f"and he has averaged {row['fantasy_points_3wk_avg']:.1f} points over the last three weeks.",
    ]
    if row['is_bad_weather']:
        parts.append("Bad weather is a downside risk for this game.")
    elif row['opp_def_ppg_allowed'] and row['opp_def_ppg_allowed'] < 15:
        parts.append(f"Risk: {row['opponent']} has been stingy against {row['position']}s "
                     f"({row['opp_def_ppg_allowed']:.1f} PPR pts/game allowed).")
    return ' '.join(parts)


def make_llm():
    """Pick the best available Bedrock LLM. Preference order:
    1. GPT-5.6 Terra — Responses API on the bedrock-mantle endpoint (the only
       API GPT-5.6 supports). Used automatically once the account's marketplace
       subscription for it finishes provisioning.
    2. gpt-oss-120b — OpenAI open-weight model via the Converse API.
    3. None — caller falls back to offline template insights.
    Returns (model_name, generate_fn) where generate_fn: prompt -> text."""
    try:
        from openai import OpenAI
        token = os.environ.get('AWS_BEARER_TOKEN_BEDROCK')
        if not token:
            from aws_bedrock_token_generator import provide_token
            token = provide_token(region=BEDROCK_REGION)
        client = OpenAI(
            base_url=f'https://bedrock-mantle.{BEDROCK_REGION}.api.aws/openai/v1',
            api_key=token,
        )
        client.responses.create(  # smoke test: fail fast if access missing
            model=BEDROCK_MODEL_ID, input='hi', max_output_tokens=16, store=False,
        )

        def gen(prompt):
            resp = client.responses.create(
                model=BEDROCK_MODEL_ID,
                input=prompt,
                reasoning={'effort': 'low'},  # short factual blurb; no deep reasoning needed
                max_output_tokens=500,
                store=False,
            )
            return resp.output_text.strip()

        return BEDROCK_MODEL_ID, gen
    except Exception as e:
        print(f"{BEDROCK_MODEL_ID} unavailable ({type(e).__name__}) — trying {FALLBACK_MODEL_ID}...")

    try:
        import boto3
        rt = boto3.client('bedrock-runtime', region_name=BEDROCK_REGION)
        rt.converse(
            modelId=FALLBACK_MODEL_ID,
            messages=[{'role': 'user', 'content': [{'text': 'hi'}]}],
            inferenceConfig={'maxTokens': 16},
        )

        def gen(prompt):
            r = rt.converse(
                modelId=FALLBACK_MODEL_ID,
                messages=[{'role': 'user', 'content': [{'text': prompt}]}],
                inferenceConfig={'maxTokens': 600, 'temperature': 0.4},
            )
            # gpt-oss responses interleave reasoning blocks with text blocks
            blocks = r['output']['message']['content']
            return ' '.join(b['text'] for b in blocks if b.get('text')).strip()

        return FALLBACK_MODEL_ID, gen
    except Exception as e:
        print(f"Fallback also unavailable ({type(e).__name__}) — using offline template insights.")

    return 'offline_template', None


def generate_insight(gen_fn, row):
    return gen_fn(build_prompt(row)) if gen_fn else template_insight(row)


llm_name, gen_fn = make_llm()
print(f"Insight generator: {llm_name}")
top = predictions_df.head(N_INSIGHTS).copy()
top['insight'] = [generate_insight(gen_fn, row) for _, row in top.iterrows()]
top['insight_source'] = llm_name

# Final output: predictions + insights table (Delta table in Databricks)
final = predictions_df.merge(
    top[['player_id', 'insight', 'insight_source']], on='player_id', how='left'
)
final.to_parquet('../data/predictions_with_insights.parquet', index=False)
print(f"Saved predictions + insights to ../data/predictions_with_insights.parquet\n")

for _, row in top.head(5).iterrows():
    print(f"--- {row['player_name']} ({row['position']}, {row['recent_team']}) "
          f"proj {row['projected_ppr']} | actual {row['actual_ppr']} ---")
    print(row['insight'], '\n')

openai.gpt-5.6-terra unavailable (AuthenticationError) — trying openai.gpt-oss-120b-1:0...
Fallback also unavailable (ModuleNotFoundError) — using offline template insights.
Insight generator: offline_template
Saved predictions + insights to ../data/predictions_with_insights.parquet

--- P.Nacua (WR, LA) proj 21.6 | actual 15.7 ---
P.Nacua projects for 21.6 PPR points against ATL. Vegas implies a 28.0-point team total in a favorable game script, and he has averaged 36.7 points over the last three weeks. 

--- D.Maye (QB, NE) proj 20.7 | actual 32.4 ---
D.Maye projects for 20.7 PPR points against NYJ. Vegas implies a 27.5-point team total in a favorable game script, and he has averaged 21.5 points over the last three weeks. Bad weather is a downside risk for this game. 

--- J.Gibbs (RB, DET) proj 20.3 | actual 8.4 ---
J.Gibbs projects for 20.3 PPR points against MIN. Vegas implies a 26.2-point team total in a favorable game script, and he has averaged 23.2 points over the last three we